# packet_analyze_v2 — LLM 판정 (Colab T4 + vLLM)

로컬에서 `analyze.sh`로 만든 `report/*.evidence.json`을 LLM에 주고 판정(verdict)을 받는다.

**전제**: 런타임 → 유형 변경 → **T4 GPU** 선택.

흐름: repo clone → vLLM(Qwen2.5-7B) OpenAI 서버 부팅 → `llm_analyze.py` 실행

In [ ]:
# 1. repo clone (★ 본인 GitHub URL로 교체)
REPO = "https://github.com/<USER>/packet_analyze_v2.git"
!git clone -q $REPO /content/packet_analyze_v2
%cd /content/packet_analyze_v2
!ls report/*.evidence.json

In [ ]:
# 2. 설치 (vLLM + openai) — 몇 분 걸림
!pip install -q vllm openai

In [ ]:
# 3. vLLM OpenAI 서버 백그라운드 부팅 (Qwen2.5-7B-Instruct-AWQ, T4 15GB에 맞춤)
#    --tool-call-parser hermes : Qwen2.5 function-calling 파서
import subprocess, os
os.makedirs('/content/logs', exist_ok=True)
srv = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', 'Qwen/Qwen2.5-7B-Instruct-AWQ',
    '--max-model-len', '16384',
    '--gpu-memory-utilization', '0.9',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'hermes',
], stdout=open('/content/logs/vllm.log','w'), stderr=subprocess.STDOUT)
print('vLLM 부팅 중... (모델 다운로드 포함 3~6분)')

In [ ]:
# 4. 서버 준비 대기 (health 폴링)
import time, urllib.request
for i in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        print('✅ vLLM 준비 완료'); break
    except Exception:
        time.sleep(5)
        if i % 6 == 0: print(f'  대기 {i*5}s...')
else:
    print('❌ 타임아웃 — /content/logs/vllm.log 확인')
    !tail -30 /content/logs/vllm.log

In [ ]:
# 5. LLM 판정 실행
NAME = '2021-06-16-ISC-forensic-contest'  # report/<NAME>.evidence.json
!python scripts/llm_analyze.py $NAME \
    --base-url http://localhost:8000/v1 \
    --model Qwen/Qwen2.5-7B-Instruct-AWQ

In [ ]:
# 6. verdict 확인
import json
print(json.dumps(json.load(open(f'report/{NAME}.verdict.json')), ensure_ascii=False, indent=2))